# Student Engagement Recognition Using Deep Learning
### A Comparative Study of Frame-Level and Temporal Models on DAiSEE

**Computer Vision & Deep Learning — University of Verona, A.Y. 2025–26**

This notebook implements the complete pipeline described in the accompanying
project report: dataset preparation, frame-level and temporal (CNN+LSTM)
model training, evaluation, Grad-CAM explainability analysis, and two
follow-up experiments (full-dataset scaling and a 3-class extension).

**Structure of this notebook (mirrors the report's section numbering):**

1. Setup and Drive mounting
2. Dataset preparation (Section 4 of the report)
   - 2.1 DAiSEE extraction
   - 2.2 Manifest construction (label parsing, subject-disjoint verification)
   - 2.3 Stratified subsampling
   - 2.4 Frame sequence extraction (HDF5)
3. Model architecture and training infrastructure (Section 5)
4. Core experiments: frame-level vs. CNN+LSTM on the stratified subset (Section 6.1–6.5)
5. Grad-CAM explainability analysis (Section 6.6)
6. Follow-up experiment: full-dataset scaling (Section 6.7)
7. Follow-up experiment: uncapped class weighting (Section 6.7.1)
8. Follow-up experiment: 3-class formulation (Section 6.8)

**Note on dataset leakage:** an earlier version of this project used a
Kaggle-hosted dataset that was later found to have severe session-level
data leakage (see report Section 4.1). This notebook reflects the corrected
pipeline built on DAiSEE after that issue was diagnosed and resolved.


## 1. Setup

Mount Google Drive and verify that the raw DAiSEE archive is present.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os

PROJECT_BASE = '/content/drive/MyDrive/DLCV_Project'
ZIP_PATH = f'{PROJECT_BASE}/data/DAiSEE.zip'

assert os.path.exists(ZIP_PATH), f"Zip not found at {ZIP_PATH} -- check the exact path/filename"
size_gb = os.path.getsize(ZIP_PATH) / (1024**3)
print(f"Zip found: {ZIP_PATH}")
print(f"Size: {size_gb:.2f} GB (expected ~14.27 GB)")


### 1.1 Why we extract locally first

Colab's Drive mount has noticeably slower I/O than local disk for operations
touching thousands of small files (such as unzipping ~18,000 archive
entries). We copy the zip to local disk first, then extract there, then
copy only the small final artifacts (manifests, HDF5 sequence files) back
to Drive for persistence across sessions.

**Known issue:** one file in the official DAiSEE archive
(`DAiSEE/DataSet/Train/210055/2100552061/2100552061.avi`) has a corrupted
CRC and cannot be extracted. It has no corresponding row in the official
labels CSV, so its loss is immaterial — we skip it explicitly below rather
than letting the whole extraction fail.

In [ ]:
import shutil
import time

print("Copying zip from Drive to local Colab disk...")
t0 = time.time()
shutil.copy(ZIP_PATH, '/content/DAiSEE.zip')
elapsed = time.time() - t0

local_size_gb = os.path.getsize('/content/DAiSEE.zip') / (1024**3)
print(f"\nCopy complete in {elapsed:.0f}s")
print(f"Local copy size: {local_size_gb:.2f} GB")
assert abs(local_size_gb - 14.27) < 0.05, "Size mismatch after copy"
print("Size verified — matches source exactly")


In [ ]:
import zipfile

print("Extracting locally, skipping the one known-corrupt file...")
t0 = time.time()

KNOWN_BAD_FILE = 'DAiSEE/DataSet/Train/210055/2100552061/2100552061.avi'
skipped = []
extracted_count = 0

with zipfile.ZipFile('/content/DAiSEE.zip', 'r') as zf:
    namelist = zf.namelist()
    total = len(namelist)

    for i, name in enumerate(namelist):
        try:
            zf.extract(name, '/content/DAiSEE_extracted')
            extracted_count += 1
        except (zipfile.BadZipFile, OSError) as e:
            skipped.append((name, str(e)))
            print(f"  SKIPPED (corrupt): {name}")

        if (i + 1) % 2000 == 0:
            elapsed = time.time() - t0
            print(f"  Progress: {i+1}/{total} ({elapsed:.0f}s elapsed)")

elapsed = time.time() - t0
print(f"\nExtraction complete in {elapsed:.0f}s ({elapsed/60:.1f} min)")
print(f"Successfully extracted: {extracted_count}")
print(f"Skipped (corrupt): {len(skipped)}")
for name, err in skipped:
    print(f"  {name}")


In [ ]:
# Sanity check: confirm extraction is complete (no empty participant
# folders -- this caught a real partial-extraction bug during development)
EXTRACTED_BASE = '/content/DAiSEE_extracted/DAiSEE'

import subprocess
result = subprocess.run(['du', '-sh', EXTRACTED_BASE], capture_output=True, text=True)
print(f"Total extracted size: {result.stdout.strip()}")

for split in ['Train', 'Validation', 'Test']:
    split_dir = f'{EXTRACTED_BASE}/DataSet/{split}'
    participants = os.listdir(split_dir)
    empty_count = sum(1 for p in participants
                       if os.path.isdir(os.path.join(split_dir, p))
                       and len(os.listdir(os.path.join(split_dir, p))) == 0)
    print(f"{split}: {len(participants)} participant folders, {empty_count} empty")


## 2. Dataset Preparation

### 2.1 Building the Manifest

We parse DAiSEE's official label CSVs and join them to the actual video
file paths on disk. Two real data-quality issues were discovered and fixed
here during development:

1. **Trailing whitespace in the `Frustration ` column name** in the official
   CSV — fixed by stripping all column names on load.
2. **~9% of clips are distributed as `.mp4` instead of `.avi`**, even though
   the labels CSV always references clips with a `.avi` suffix. We match
   clips by base filename (ignoring extension) rather than assuming a fixed
   extension, which correctly recovers these clips.

The binary `engagement_binary` label collapses DAiSEE's 4-level ordinal
Engagement score (0=very low ... 3=very high) to a binary label:
levels 0–1 → disengaged, levels 2–3 → engaged.

In [ ]:
import pandas as pd

def normalize_columns(df):
    """Strips whitespace from column names (fixes the 'Frustration ' bug)."""
    df.columns = [c.strip() for c in df.columns]
    return df


def build_clip_path_index(dataset_split_dir):
    """
    Walks DataSet/<Split>/<ParticipantID>/<ClipID>/<video_file> and returns
    a dict keyed by BASE clip ID (no extension), since ~9% of clips are
    distributed as .mp4 rather than .avi despite the labels CSV always
    using a .avi suffix.
    """
    VALID_EXTENSIONS = ('.avi', '.mp4')
    index = {}
    for participant_id in os.listdir(dataset_split_dir):
        participant_path = os.path.join(dataset_split_dir, participant_id)
        if not os.path.isdir(participant_path):
            continue
        for clip_folder in os.listdir(participant_path):
            clip_path_dir = os.path.join(participant_path, clip_folder)
            if not os.path.isdir(clip_path_dir):
                continue
            for fname in os.listdir(clip_path_dir):
                ext = os.path.splitext(fname)[1].lower()
                if ext in VALID_EXTENSIONS:
                    base_id = os.path.splitext(fname)[0]
                    index[base_id] = {"path": os.path.join(clip_path_dir, fname),
                                       "participant_id": participant_id,
                                       "actual_extension": ext}
    return index


def build_daisee_manifest(base_dir, split_name, engagement_collapse=True):
    """
    Joins the official <Split>Labels.csv to actual video file paths.
    Returns (manifest_df, missing_files) -- missing_files lists any
    labeled clip that could not be matched to a file on disk (should be
    empty given the extension-matching fix above; any nonzero count
    indicates a genuine data problem worth investigating).
    """
    label_path = os.path.join(base_dir, "Labels", f"{split_name}Labels.csv")
    dataset_split_dir = os.path.join(base_dir, "DataSet", split_name)

    labels_df = pd.read_csv(label_path)
    labels_df = normalize_columns(labels_df)
    clip_index = build_clip_path_index(dataset_split_dir)

    records, missing_files = [], []
    for _, row in labels_df.iterrows():
        clip_id_csv = row["ClipID"]
        clip_base_id = os.path.splitext(clip_id_csv)[0]

        if clip_base_id not in clip_index:
            missing_files.append(clip_id_csv)
            continue

        record = {
            "clip_id": clip_base_id,
            "video_path": clip_index[clip_base_id]["path"],
            "participant_id": clip_index[clip_base_id]["participant_id"],
            "split": split_name,
            "boredom": int(row["Boredom"]),
            "engagement": int(row["Engagement"]),
            "confusion": int(row["Confusion"]),
            "frustration": int(row["Frustration"]),
        }
        if engagement_collapse:
            record["engagement_binary"] = 1 if row["Engagement"] >= 2 else 0
        records.append(record)

    manifest_df = pd.DataFrame(records)
    print(f"[{split_name}] Labels in CSV: {len(labels_df)}")
    print(f"[{split_name}] Matched to video files: {len(manifest_df)}")
    print(f"[{split_name}] Labeled clips missing video file: {len(missing_files)}")
    if missing_files:
        print(f"[{split_name}]   Sample missing: {missing_files[:5]}")
    return manifest_df, missing_files


train_manifest, train_missing = build_daisee_manifest(EXTRACTED_BASE, "Train")
val_manifest, val_missing = build_daisee_manifest(EXTRACTED_BASE, "Validation")
test_manifest, test_missing = build_daisee_manifest(EXTRACTED_BASE, "Test")


### 2.2 Subject-Disjoint Verification

This is the methodological safeguard motivated directly by the data
leakage discovered in our earlier Kaggle-dataset attempt (report Section
4.1): we explicitly verify that no participant appears in more than one
split, since DAiSEE's official split is designed to be subject-disjoint
but we do not assume this without checking.

In [ ]:
def verify_subject_disjoint(train_df, val_df, test_df):
    tp = set(train_df["participant_id"])
    vp = set(val_df["participant_id"])
    tsp = set(test_df["participant_id"])
    tv, tt, vt = tp & vp, tp & tsp, vp & tsp

    print(f"Train participants: {len(tp)}  Val: {len(vp)}  Test: {len(tsp)}")
    print(f"Train/Val overlap:  {len(tv)} {'<-- LEAK!' if tv else '(clean)'}")
    print(f"Train/Test overlap: {len(tt)} {'<-- LEAK!' if tt else '(clean)'}")
    print(f"Val/Test overlap:   {len(vt)} {'<-- LEAK!' if vt else '(clean)'}")
    return {"train_val_overlap": tv, "train_test_overlap": tt, "val_test_overlap": vt}


overlaps = verify_subject_disjoint(train_manifest, val_manifest, test_manifest)

print(f"\nFinal sizes: train={len(train_manifest)}, val={len(val_manifest)}, test={len(test_manifest)}")

# Persist the manifests to Drive so this expensive extraction step never
# needs to be repeated in future sessions
train_manifest.to_csv(f"{PROJECT_BASE}/daisee_train_manifest.csv", index=False)
val_manifest.to_csv(f"{PROJECT_BASE}/daisee_val_manifest.csv", index=False)
test_manifest.to_csv(f"{PROJECT_BASE}/daisee_test_manifest.csv", index=False)
print("Manifests saved to Drive.")


### 2.3 Stratified Subsampling

DAiSEE's natural class distribution is severely imbalanced (~95% engaged
at the clip level). Given the four-week project timeline, we work with a
stratified subsample rather than the full ~8,500-clip dataset, preserving
**every** minority-class (disengaged) clip and capping the majority class
at a 3:1 ratio. A per-participant cap (15 clips) prevents any single
prolific subject from dominating the subsample.

In [ ]:
import random

def stratified_subset(manifest_df, target_ratio=3.0, max_clips_per_participant=15, seed=42):
    """
    Keeps ALL minority-class clips, subsamples the majority class down to
    `target_ratio`, with a per-participant cap applied to both classes.
    """
    rng = random.Random(seed)

    minority_df = manifest_df[manifest_df["engagement_binary"] == 0].copy()
    majority_df = manifest_df[manifest_df["engagement_binary"] == 1].copy()

    def cap_per_participant(df, max_per_participant):
        kept_indices = []
        for participant_id, group in df.groupby("participant_id"):
            indices = list(group.index)
            rng.shuffle(indices)
            kept_indices.extend(indices[:max_per_participant])
        return df.loc[kept_indices]

    minority_capped = cap_per_participant(minority_df, max_clips_per_participant)
    majority_capped = cap_per_participant(majority_df, max_clips_per_participant)

    target_majority_count = int(len(minority_capped) * target_ratio)
    if len(majority_capped) > target_majority_count:
        majority_indices = list(majority_capped.index)
        rng.shuffle(majority_indices)
        majority_sample = majority_capped.loc[majority_indices[:target_majority_count]]
    else:
        # Not enough majority clips to hit the target ratio under the cap --
        # use everything available (this happens for the validation split,
        # which has fewer participants; see report Section 4.3)
        majority_sample = majority_capped

    subset_df = pd.concat([minority_capped, majority_sample]).reset_index(drop=True)

    report = {
        "original_total": len(manifest_df),
        "subset_total": len(subset_df),
        "subset_minority": len(subset_df[subset_df["engagement_binary"] == 0]),
        "subset_majority": len(subset_df[subset_df["engagement_binary"] == 1]),
        "achieved_ratio": (len(subset_df[subset_df["engagement_binary"] == 1]) /
                           max(len(subset_df[subset_df["engagement_binary"] == 0]), 1)),
        "num_participants_in_subset": subset_df["participant_id"].nunique(),
    }
    return subset_df, report


train_subset, train_report = stratified_subset(train_manifest, seed=42)
val_subset, val_report = stratified_subset(val_manifest, seed=42)
test_subset, test_report = stratified_subset(test_manifest, seed=42)

for name, report in [("Train", train_report), ("Val", val_report), ("Test", test_report)]:
    print(f"{name}: {report}")

train_subset.to_csv(f"{PROJECT_BASE}/daisee_train_subset.csv", index=False)
val_subset.to_csv(f"{PROJECT_BASE}/daisee_val_subset.csv", index=False)
test_subset.to_csv(f"{PROJECT_BASE}/daisee_test_subset.csv", index=False)
print("\nStratified subsets saved to Drive.")


### 2.4 Frame Sequence Extraction

We extract overlapping 8-frame sequences from each clip (112×112 resolution)
using a sliding window with **stride 16**. This stride was chosen after an
earlier attempt at stride 4 produced an HDF5 file larger than the raw
dataset itself (18GB+) due to massive frame overlap; stride 16 reduces this
to a manageable ~5.3GB while still providing ~18 sequences per 10-second
clip.

In [ ]:
!pip install h5py -q

import h5py
import numpy as np
import cv2

def extract_sequences(video_path, sequence_length=8, stride=16, resize=(112, 112)):
    """Extracts overlapping frame sequences from a video via sliding window."""
    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened():
        raise IOError(f"Could not open video: {video_path}")
    frames = []
    while True:
        ret, frame = cap.read()
        if not ret:
            break
        frame = cv2.resize(frame, resize)
        frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        frames.append(frame)
    cap.release()

    total_frames = len(frames)
    if total_frames < sequence_length:
        return np.empty((0, sequence_length, resize[1], resize[0], 3), dtype=np.uint8)

    sequences = []
    for start in range(0, total_frames - sequence_length + 1, stride):
        sequences.append(frames[start:start + sequence_length])
    return np.array(sequences, dtype=np.uint8)


def build_sequence_hdf5(manifest_df, output_h5_path, sequence_length=8, stride=16,
                          resize=(112, 112), progress_every=50):
    """
    Extracts sequences for every clip in manifest_df and writes them to an
    HDF5 file. Resumable: re-running with the same output path skips clips
    that already have a dataset entry, so an interrupted run can continue
    rather than starting over.
    """
    mode = 'a' if os.path.exists(output_h5_path) else 'w'
    report = {"processed": 0, "skipped_already_done": 0, "failed": [], "total_sequences_written": 0}

    with h5py.File(output_h5_path, mode) as hf:
        seq_group = hf['sequences'] if 'sequences' in hf else hf.create_group('sequences')

        for i, row in manifest_df.iterrows():
            clip_id = row['clip_id']
            if clip_id in seq_group:
                report["skipped_already_done"] += 1
                continue
            try:
                sequences = extract_sequences(row['video_path'], sequence_length, stride, resize)
                if sequences.shape[0] == 0:
                    report["failed"].append((clip_id, "too short, 0 sequences"))
                    continue
                seq_group.create_dataset(clip_id, data=sequences, compression='gzip', compression_opts=4)
                report["processed"] += 1
                report["total_sequences_written"] += sequences.shape[0]
            except Exception as e:
                report["failed"].append((clip_id, str(e)))

            if (i + 1) % progress_every == 0:
                n_proc = report["processed"]
                n_fail = len(report["failed"])
                print(f"  Progress: {i+1}/{len(manifest_df)} (processed={n_proc}, failed={n_fail})")

    return report


def write_metadata_table(manifest_df, output_h5_path):
    """Writes clip-level labels to the SAME HDF5 file for fast lookup."""
    with h5py.File(output_h5_path, 'a') as hf:
        if 'metadata' in hf:
            del hf['metadata']
        meta_group = hf.create_group('metadata')
        string_dtype = h5py.special_dtype(vlen=str)
        meta_group.create_dataset('clip_id', data=manifest_df['clip_id'].astype(str).values, dtype=string_dtype)
        meta_group.create_dataset('participant_id', data=manifest_df['participant_id'].astype(str).values, dtype=string_dtype)
        meta_group.create_dataset('engagement_binary', data=manifest_df['engagement_binary'].values)
        meta_group.create_dataset('engagement', data=manifest_df['engagement'].values)
        meta_group.create_dataset('boredom', data=manifest_df['boredom'].values)
        meta_group.create_dataset('confusion', data=manifest_df['confusion'].values)
        meta_group.create_dataset('frustration', data=manifest_df['frustration'].values)


for name, subset_df in [("train", train_subset), ("val", val_subset), ("test", test_subset)]:
    output_path = f"/content/daisee_{name}_sequences.h5"
    print(f"\nExtracting sequences for {name} ({len(subset_df)} clips), stride=16")
    report = build_sequence_hdf5(subset_df, output_path, sequence_length=8, stride=16)
    write_metadata_table(subset_df, output_path)
    size_mb = os.path.getsize(output_path) / (1024**2)
    print(f"{name} report: {report}")
    print(f"HDF5 file size: {size_mb:.1f} MB")


In [ ]:
# Persist the compact HDF5 sequence files to Drive
for name in ["train", "val", "test"]:
    src = f"/content/daisee_{name}_sequences.h5"
    dst = f"{PROJECT_BASE}/daisee_{name}_sequences.h5"
    shutil.copy(src, dst)
    size_mb = os.path.getsize(dst) / (1024**2)
    print(f"Saved {name}: {size_mb:.1f} MB")


## 3. Dataset and Model Infrastructure

### 3.1 PyTorch Dataset Class

Each item is one 8-frame sequence with its clip-level binary label. A single
clip contributes multiple sequences (overlapping windows), so the dataset's
length is the total sequence count, not the clip count.

**Important implementation detail:** standard torchvision augmentations
(random flip, color jitter) sample their randomness independently every
call, which is correct for single images but WRONG for video sequences —
applying an independent random flip to each frame would flip some frames in
a sequence but not others, producing a physically meaningless signal (a
face flipping orientation mid-clip). `SequenceConsistentTransform` below
samples augmentation parameters ONCE per sequence and applies them
identically to every frame.

In [ ]:
import h5py
import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as T

IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]


class DAiSEESequenceDataset(Dataset):
    """Binary-labeled dataset (engagement_binary: 0=disengaged, 1=engaged)."""

    def __init__(self, h5_path, transform=None):
        self.h5_path = h5_path
        self.transform = transform
        self._index = []
        with h5py.File(h5_path, 'r') as hf:
            seq_group = hf['sequences']
            meta = hf['metadata']
            clip_ids = [c if isinstance(c, str) else c.decode() for c in meta['clip_id'][:]]
            labels = meta['engagement_binary'][:]
            label_lookup = dict(zip(clip_ids, labels))
            for clip_id in seq_group.keys():
                num_sequences = seq_group[clip_id].shape[0]
                label = int(label_lookup[clip_id])
                for seq_idx in range(num_sequences):
                    self._index.append((clip_id, seq_idx, label))

    def __len__(self):
        return len(self._index)

    def __getitem__(self, idx):
        clip_id, seq_idx, label = self._index[idx]
        with h5py.File(self.h5_path, 'r') as hf:
            sequence = hf['sequences'][clip_id][seq_idx]
        if self.transform:
            sequence_tensor = self.transform(sequence)
        else:
            sequence_tensor = torch.from_numpy(sequence).float() / 255.0
            sequence_tensor = sequence_tensor.permute(0, 3, 1, 2)
        return sequence_tensor, label

    def get_label_distribution(self):
        labels = [item[2] for item in self._index]
        unique, counts = np.unique(labels, return_counts=True)
        return dict(zip(unique.tolist(), counts.tolist()))


class SequenceConsistentTransform:
    """Applies the SAME augmentation decision to every frame in a sequence."""

    def __init__(self, train=True):
        self.train = train
        self.to_tensor = T.ToTensor()
        self.normalize = T.Normalize(IMAGENET_MEAN, IMAGENET_STD)

    def __call__(self, sequence):
        seq_len = sequence.shape[0]
        do_flip = self.train and (np.random.random() < 0.5)
        brightness_factor = np.random.uniform(0.8, 1.2) if self.train else 1.0
        contrast_factor = np.random.uniform(0.8, 1.2) if self.train else 1.0

        frames_out = []
        for i in range(seq_len):
            frame = sequence[i]
            if do_flip:
                frame = np.ascontiguousarray(frame[:, ::-1, :])
            frame_tensor = self.to_tensor(frame)
            if self.train:
                frame_tensor = torch.clamp(frame_tensor * brightness_factor, 0, 1)
                mean_val = frame_tensor.mean()
                frame_tensor = torch.clamp((frame_tensor - mean_val) * contrast_factor + mean_val, 0, 1)
            frame_tensor = self.normalize(frame_tensor)
            frames_out.append(frame_tensor)
        return torch.stack(frames_out)


def build_frame_transform(train=True):
    return SequenceConsistentTransform(train=train)


### 3.2 CNN+LSTM Architecture

Each frame in a sequence is passed through a shared CNN backbone
(MobileNetV3-Large or ResNet-18, both ImageNet-pretrained) to produce a
feature vector. The sequence of feature vectors is then processed by a
single-layer LSTM, and the final hidden state is passed through a dropout
layer and linear classification head.

In [ ]:
import torch.nn as nn
import torchvision.models as models


class CNNLSTMClassifier(nn.Module):
    def __init__(self, backbone="mobilenetv3", num_classes=2, lstm_hidden_size=256,
                 lstm_num_layers=1, dropout=0.5, freeze_backbone=False, pretrained=True):
        super().__init__()
        if backbone == "mobilenetv3":
            weights = models.MobileNet_V3_Large_Weights.IMAGENET1K_V2 if pretrained else None
            cnn = models.mobilenet_v3_large(weights=weights)
            self.cnn_backbone = cnn.features
            self.cnn_pool = nn.AdaptiveAvgPool2d(1)
            feature_dim = 960
        elif backbone == "resnet18":
            weights = models.ResNet18_Weights.IMAGENET1K_V1 if pretrained else None
            cnn = models.resnet18(weights=weights)
            self.cnn_backbone = nn.Sequential(*list(cnn.children())[:-2])
            self.cnn_pool = nn.AdaptiveAvgPool2d(1)
            feature_dim = 512
        else:
            raise ValueError(f"Unknown backbone: {backbone}")

        if freeze_backbone:
            for param in self.cnn_backbone.parameters():
                param.requires_grad = False

        self.feature_dim = feature_dim
        self.lstm = nn.LSTM(input_size=feature_dim, hidden_size=lstm_hidden_size,
                             num_layers=lstm_num_layers, batch_first=True,
                             dropout=dropout if lstm_num_layers > 1 else 0.0)
        self.classifier = nn.Sequential(nn.Dropout(dropout), nn.Linear(lstm_hidden_size, num_classes))

    def forward(self, x):
        batch_size, seq_len, C, H, W = x.shape
        x = x.view(batch_size * seq_len, C, H, W)
        features = self.cnn_backbone(x)
        features = self.cnn_pool(features)
        features = features.view(batch_size, seq_len, self.feature_dim)
        lstm_out, (h_n, c_n) = self.lstm(features)
        final_hidden = h_n[-1]
        logits = self.classifier(final_hidden)
        return logits


def build_cnn_lstm(backbone="mobilenetv3", num_classes=2, pretrained=True, **kwargs):
    return CNNLSTMClassifier(backbone=backbone, num_classes=num_classes, pretrained=pretrained, **kwargs)


def build_frame_classifier(num_classes=2):
    """Plain MobileNetV3 classifier with no temporal modeling -- the
    frame-level baseline used throughout Section 6 of the report."""
    model = models.mobilenet_v3_large(weights=models.MobileNet_V3_Large_Weights.IMAGENET1K_V2)
    in_features = model.classifier[-1].in_features
    model.classifier[-1] = nn.Linear(in_features, num_classes)
    return model


### 3.3 Loss Function and Training Loop

We use categorical cross-entropy with inverse-frequency class weighting,
optionally capped at a maximum multiplier (used in the full-dataset
follow-up experiment in Section 7, where the natural class imbalance is
severe enough that the raw inverse-frequency ratio risks destabilizing
training).

In [ ]:
from sklearn.metrics import f1_score, classification_report, confusion_matrix
import pandas as pd
import time

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {DEVICE}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

CHECKPOINT_DIR = f"{PROJECT_BASE}/checkpoints"
LOG_DIR = f"{PROJECT_BASE}/logs"
os.makedirs(CHECKPOINT_DIR, exist_ok=True)
os.makedirs(LOG_DIR, exist_ok=True)


def compute_capped_class_weights(dist, max_multiplier=5.0):
    """
    Computes inverse-frequency class weights for a BINARY {0, 1} distribution,
    capped at max_multiplier. A cap of 5.0 means the minority class is
    weighted at most 5x more than the majority class, even if the true
    imbalance ratio is more extreme (see report Section 6.7 for why this
    matters and Section 6.7.1 for what happens when the cap is removed).
    """
    total = sum(dist.values())
    raw_weights = torch.tensor([total / dist[0], total / dist[1]], dtype=torch.float32)
    normalized = raw_weights / raw_weights.min()
    capped = torch.clamp(normalized, max=max_multiplier)
    final = capped / capped.sum() * 2
    return final


def train_cnn_lstm_capped(model, model_name, train_dataset, val_dataset,
                            num_epochs=15, lr=1e-4, weight_decay=1e-4,
                            batch_size=16, max_weight_multiplier=5.0, patience=4):
    """
    General training loop for binary CNN+LSTM models. Saves the
    best-validation-macro-F1 checkpoint and a per-epoch CSV log.
    """
    model = model.to(DEVICE)
    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=2)
    val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False, num_workers=2)

    dist = train_dataset.get_label_distribution()
    weights = compute_capped_class_weights(dist, max_weight_multiplier).to(DEVICE)
    print(f"Class distribution: {dist}")
    print(f"Class weights (max {max_weight_multiplier}x): {weights.cpu().numpy()}")

    criterion = nn.CrossEntropyLoss(weight=weights)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)

    log_path = f"{LOG_DIR}/{model_name}_training_log.csv"
    checkpoint_path = f"{CHECKPOINT_DIR}/{model_name}_best.pth"

    best_val_f1 = 0.0
    epochs_no_improve = 0
    log_rows = []

    for epoch in range(1, num_epochs + 1):
        t0 = time.time()
        model.train()
        train_loss = 0.0
        for sequences, labels in train_loader:
            sequences, labels = sequences.to(DEVICE), labels.to(DEVICE)
            optimizer.zero_grad()
            outputs = model(sequences)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            train_loss += loss.item() * sequences.size(0)
        train_loss /= len(train_loader.dataset)

        model.eval()
        val_loss = 0.0
        all_preds, all_labels = [], []
        with torch.no_grad():
            for sequences, labels in val_loader:
                sequences, labels = sequences.to(DEVICE), labels.to(DEVICE)
                outputs = model(sequences)
                loss = criterion(outputs, labels)
                val_loss += loss.item() * sequences.size(0)
                preds = outputs.argmax(dim=1)
                all_preds.extend(preds.cpu().numpy())
                all_labels.extend(labels.cpu().numpy())
        val_loss /= len(val_loader.dataset)
        val_f1 = f1_score(all_labels, all_preds, average="macro")

        elapsed = time.time() - t0
        print(f"[{model_name}] Epoch {epoch:02d}/{num_epochs} | "
              f"train_loss={train_loss:.4f} val_loss={val_loss:.4f} "
              f"val_macro_f1={val_f1:.4f} ({elapsed:.1f}s)")

        log_rows.append({"epoch": epoch, "train_loss": train_loss,
                          "val_loss": val_loss, "val_macro_f1": val_f1})
        pd.DataFrame(log_rows).to_csv(log_path, index=False)

        if val_f1 > best_val_f1:
            best_val_f1 = val_f1
            epochs_no_improve = 0
            torch.save({"model_state_dict": model.state_dict(),
                        "epoch": epoch, "val_f1": val_f1}, checkpoint_path)
            print(f"  -> New best model saved (val_macro_f1={val_f1:.4f})")
        else:
            epochs_no_improve += 1
            if epochs_no_improve >= patience:
                print(f"  -> Early stopping at epoch {epoch}")
                break

    print(f"\n[{model_name}] Best validation macro-F1: {best_val_f1:.4f}")
    return checkpoint_path, best_val_f1


def evaluate_on_test(model, checkpoint_path, test_dataset, model_name,
                       class_names=("disengaged", "engaged"), batch_size=32):
    """Loads the best checkpoint and reports full test-set metrics."""
    checkpoint = torch.load(checkpoint_path, map_location=DEVICE)
    if isinstance(checkpoint, dict) and "model_state_dict" in checkpoint:
        model.load_state_dict(checkpoint["model_state_dict"])
    else:
        model.load_state_dict(checkpoint)
    model = model.to(DEVICE)
    model.eval()

    test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False, num_workers=2)

    all_preds, all_labels = [], []
    with torch.no_grad():
        for sequences, labels in test_loader:
            sequences = sequences.to(DEVICE)
            outputs = model(sequences)
            preds = outputs.argmax(dim=1)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.numpy())

    acc = (np.array(all_preds) == np.array(all_labels)).mean()
    macro_f1 = f1_score(all_labels, all_preds, average="macro")
    cm = confusion_matrix(all_labels, all_preds)

    print(f"\n{'='*60}")
    print(f"{model_name} — TEST SET RESULTS")
    print(f"{'='*60}")
    print(f"Accuracy: {acc:.4f}")
    print(f"Macro-F1: {macro_f1:.4f}")
    print(f"\nClassification report:")
    print(classification_report(all_labels, all_preds, target_names=list(class_names), digits=3))
    print(f"\nConfusion matrix:")
    print(pd.DataFrame(cm, index=list(class_names), columns=list(class_names)))

    return {"model_name": model_name, "test_accuracy": acc, "test_macro_f1": macro_f1,
            "confusion_matrix": cm.tolist()}


## 4. Core Experiment: Frame-Level vs. CNN+LSTM (Stratified Subset)

This is the central comparison of the project, corresponding to report
Sections 6.1–6.5: three models trained on the stratified subset built in
Part 1 — a frame-level MobileNetV3 baseline, MobileNetV3+LSTM, and
ResNet-18+LSTM.

In [ ]:
# Load the stratified-subset datasets built in Part 1
train_transform = build_frame_transform(train=True)
eval_transform = build_frame_transform(train=False)

train_dataset = DAiSEESequenceDataset(f"{PROJECT_BASE}/daisee_train_sequences.h5", transform=train_transform)
val_dataset = DAiSEESequenceDataset(f"{PROJECT_BASE}/daisee_val_sequences.h5", transform=eval_transform)
test_dataset = DAiSEESequenceDataset(f"{PROJECT_BASE}/daisee_test_sequences.h5", transform=eval_transform)

print(f"Train: {len(train_dataset)} sequences, distribution: {train_dataset.get_label_distribution()}")
print(f"Val:   {len(val_dataset)} sequences, distribution: {val_dataset.get_label_distribution()}")
print(f"Test:  {len(test_dataset)} sequences, distribution: {test_dataset.get_label_distribution()}")


### 4.1 Frame-Level Baseline (MobileNetV3, No Temporal Modeling)

In [ ]:
class DAiSEEFrameDataset(Dataset):
    """
    Wraps the SAME underlying HDF5 sequences, but treats each frame within
    a sequence as an INDEPENDENT training sample (no temporal modeling).
    Used to isolate whether the CNN backbone alone can learn the task,
    decoupled from any LSTM contribution.
    """
    def __init__(self, h5_path, transform=None):
        self.h5_path = h5_path
        self.transform = transform
        self._index = []
        with h5py.File(h5_path, 'r') as hf:
            seq_group = hf['sequences']
            meta = hf['metadata']
            clip_ids = [c if isinstance(c, str) else c.decode() for c in meta['clip_id'][:]]
            labels = meta['engagement_binary'][:]
            label_lookup = dict(zip(clip_ids, labels))
            for clip_id in seq_group.keys():
                num_sequences = seq_group[clip_id].shape[0]
                label = int(label_lookup[clip_id])
                for seq_idx in range(num_sequences):
                    self._index.append((clip_id, seq_idx, 0, label))  # always use first frame

    def __len__(self):
        return len(self._index)

    def __getitem__(self, idx):
        clip_id, seq_idx, frame_idx, label = self._index[idx]
        with h5py.File(self.h5_path, 'r') as hf:
            frame = hf['sequences'][clip_id][seq_idx][frame_idx]
        if self.transform:
            frame_tensor = self.transform(frame)
        else:
            frame_tensor = torch.from_numpy(frame).float().permute(2, 0, 1) / 255.0
        return frame_tensor, label


def build_single_frame_transform(train=True):
    return T.Compose([T.ToTensor(), T.Normalize(IMAGENET_MEAN, IMAGENET_STD)])


frame_train_dataset = DAiSEEFrameDataset(f"{PROJECT_BASE}/daisee_train_sequences.h5",
                                           transform=build_single_frame_transform(train=True))
frame_val_dataset = DAiSEEFrameDataset(f"{PROJECT_BASE}/daisee_val_sequences.h5",
                                         transform=build_single_frame_transform(train=False))
frame_test_dataset = DAiSEEFrameDataset(f"{PROJECT_BASE}/daisee_test_sequences.h5",
                                          transform=build_single_frame_transform(train=False))

print(f"Frame-level train: {len(frame_train_dataset)} samples")
print(f"Frame-level val: {len(frame_val_dataset)} samples")


In [ ]:
def train_frame_classifier(model, model_name, train_dataset, val_dataset,
                             num_epochs=10, lr=1e-4, batch_size=32, patience=3):
    """Training loop for the frame-level (non-temporal) baseline."""
    model = model.to(DEVICE)
    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=2)
    val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False, num_workers=2)

    dist_0 = sum(1 for item in train_dataset._index if item[3] == 0)
    dist_1 = sum(1 for item in train_dataset._index if item[3] == 1)
    total = dist_0 + dist_1
    weights = torch.tensor([total / dist_0, total / dist_1], dtype=torch.float32)
    weights = (weights / weights.sum() * 2).to(DEVICE)
    print(f"Class weights: {weights.cpu().numpy()}")

    criterion = nn.CrossEntropyLoss(weight=weights)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=1e-4)

    best_val_f1 = 0.0
    epochs_no_improve = 0

    for epoch in range(1, num_epochs + 1):
        t0 = time.time()
        model.train()
        train_loss = 0.0
        for images, labels in train_loader:
            images, labels = images.to(DEVICE), labels.to(DEVICE)
            optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            train_loss += loss.item() * images.size(0)
        train_loss /= len(train_loader.dataset)

        model.eval()
        val_loss = 0.0
        all_preds, all_labels = [], []
        with torch.no_grad():
            for images, labels in val_loader:
                images, labels = images.to(DEVICE), labels.to(DEVICE)
                outputs = model(images)
                loss = criterion(outputs, labels)
                val_loss += loss.item() * images.size(0)
                preds = outputs.argmax(dim=1)
                all_preds.extend(preds.cpu().numpy())
                all_labels.extend(labels.cpu().numpy())
        val_loss /= len(val_loader.dataset)
        val_f1 = f1_score(all_labels, all_preds, average="macro")

        elapsed = time.time() - t0
        print(f"[{model_name}] Epoch {epoch:02d}/{num_epochs} | "
              f"train_loss={train_loss:.4f} val_loss={val_loss:.4f} val_macro_f1={val_f1:.4f} ({elapsed:.1f}s)")

        if val_f1 > best_val_f1:
            best_val_f1 = val_f1
            epochs_no_improve = 0
            torch.save(model.state_dict(), f"{CHECKPOINT_DIR}/{model_name}_best.pth")
            print(f"  -> New best (val_macro_f1={val_f1:.4f})")
        else:
            epochs_no_improve += 1
            if epochs_no_improve >= patience:
                print(f"  -> Early stopping at epoch {epoch}")
                break

    print(f"\n[{model_name}] Best val macro-F1: {best_val_f1:.4f}")
    return best_val_f1


frame_model = build_frame_classifier()
best_f1_frame = train_frame_classifier(frame_model, "mobilenetv3_framelevel",
                                          frame_train_dataset, frame_val_dataset,
                                          num_epochs=10, batch_size=32)


### 4.2 CNN+LSTM Models (MobileNetV3 and ResNet-18 Backbones)

In [ ]:
# MobileNetV3 + LSTM
model_mobilenet_lstm = build_cnn_lstm(backbone="mobilenetv3", num_classes=2)
ckpt_path_mob, val_f1_mob = train_cnn_lstm_capped(
    model_mobilenet_lstm, "mobilenetv3_lstm",
    train_dataset, val_dataset,
    num_epochs=15, batch_size=16
)


In [ ]:
# ResNet-18 + LSTM -- this turns out to be the best-performing configuration
# overall (see report Section 6.1)
model_resnet_lstm = build_cnn_lstm(backbone="resnet18", num_classes=2)
ckpt_path_resnet, val_f1_resnet = train_cnn_lstm_capped(
    model_resnet_lstm, "resnet18_lstm",
    train_dataset, val_dataset,
    num_epochs=15, batch_size=16, patience=4
)


### 4.3 Ablation: Frozen Backbone

We test whether freezing the CNN backbone (forcing reliance on
general-purpose ImageNet features rather than memorizing subject identity)
improves generalization. See report Section 7.4 for the discussion of why
this intervention did not improve test-set macro-F1 despite plausible
motivation.

In [ ]:
model_resnet_lstm_frozen = build_cnn_lstm(backbone="resnet18", num_classes=2, freeze_backbone=True)

total_params = sum(p.numel() for p in model_resnet_lstm_frozen.parameters())
trainable_params = sum(p.numel() for p in model_resnet_lstm_frozen.parameters() if p.requires_grad)
print(f"Total parameters: {total_params:,}")
print(f"Trainable parameters: {trainable_params:,} ({trainable_params/total_params*100:.1f}%)")

ckpt_path_frozen, val_f1_frozen = train_cnn_lstm_capped(
    model_resnet_lstm_frozen, "resnet18_lstm_frozen",
    train_dataset, val_dataset,
    num_epochs=15, batch_size=32, lr=1e-3, patience=5
)


### 4.4 Test Set Evaluation — All Four Models

In [ ]:
results = []

frame_model_eval = build_frame_classifier()
results.append(evaluate_on_test(frame_model_eval, f"{CHECKPOINT_DIR}/mobilenetv3_framelevel_best.pth",
                                  frame_test_dataset, "MobileNetV3 (frame-level)"))

model_mobilenet_lstm_eval = build_cnn_lstm(backbone="mobilenetv3", num_classes=2)
results.append(evaluate_on_test(model_mobilenet_lstm_eval, f"{CHECKPOINT_DIR}/mobilenetv3_lstm_best.pth",
                                  test_dataset, "MobileNetV3 + LSTM"))

model_resnet_lstm_eval = build_cnn_lstm(backbone="resnet18", num_classes=2)
results.append(evaluate_on_test(model_resnet_lstm_eval, f"{CHECKPOINT_DIR}/resnet18_lstm_best.pth",
                                  test_dataset, "ResNet-18 + LSTM"))

model_resnet_lstm_frozen_eval = build_cnn_lstm(backbone="resnet18", num_classes=2, freeze_backbone=True)
results.append(evaluate_on_test(model_resnet_lstm_frozen_eval, f"{CHECKPOINT_DIR}/resnet18_lstm_frozen_best.pth",
                                  test_dataset, "ResNet-18 + LSTM (frozen backbone)"))

print("\n\n" + "="*60)
print("FINAL TEST SET COMPARISON")
print("="*60)
summary_df = pd.DataFrame(results)[["model_name", "test_accuracy", "test_macro_f1"]]
print(summary_df.to_string(index=False))

summary_df.to_csv(f"{PROJECT_BASE}/final_test_results_all4.csv", index=False)


## 5. Qualitative Analysis via Grad-CAM

We adapt Grad-CAM (Selvaraju et al., 2017) to our CNN+LSTM architecture by
computing a separate spatial attention heatmap for each frame in a
sequence: the gradient of the predicted class logit is backpropagated
through the LSTM and classification head to each frame's final
convolutional feature map independently.

**Implementation note:** cuDNN's LSTM backward pass is restricted to
training mode on GPU and raises a `RuntimeError` if called during `eval()`.
We work around this by temporarily disabling cuDNN for the duration of the
Grad-CAM forward/backward pass, which allows the (still eval-mode, no
dropout) model to be explained correctly.

In [ ]:
!pip install opencv-python -q

import torch.nn.functional as F
import cv2
import matplotlib.pyplot as plt


class SequenceGradCAM:
    def __init__(self, model, target_layer_name="cnn_backbone"):
        self.model = model
        self.model.eval()
        self.activations = None
        self.gradients = None
        target_module = getattr(model, target_layer_name)
        self.forward_handle = target_module.register_forward_hook(self._save_activation)
        self.backward_handle = target_module.register_full_backward_hook(self._save_gradient)

    def _save_activation(self, module, input, output):
        self.activations = output

    def _save_gradient(self, module, grad_input, grad_output):
        self.gradients = grad_output[0]

    def remove_hooks(self):
        self.forward_handle.remove()
        self.backward_handle.remove()

    def generate(self, sequence_tensor, target_class=None):
        assert sequence_tensor.shape[0] == 1
        sequence_tensor = sequence_tensor.clone().requires_grad_(True)

        # cuDNN's LSTM backward pass is restricted to training mode on GPU --
        # disable it temporarily so backprop through the LSTM works in eval mode
        cudnn_was_enabled = torch.backends.cudnn.enabled
        torch.backends.cudnn.enabled = False
        try:
            logits = self.model(sequence_tensor)
            probs = F.softmax(logits, dim=1)
            if target_class is None:
                target_class = logits.argmax(dim=1).item()
            confidence = probs[0, target_class].item()
            self.model.zero_grad()
            score = logits[0, target_class]
            score.backward()
        finally:
            torch.backends.cudnn.enabled = cudnn_was_enabled

        batch_size, seq_len = sequence_tensor.shape[0], sequence_tensor.shape[1]
        channels = self.activations.shape[1]
        h, w = self.activations.shape[2], self.activations.shape[3]
        activations = self.activations.view(batch_size, seq_len, channels, h, w)[0]
        gradients = self.gradients.view(batch_size, seq_len, channels, h, w)[0]
        alpha = gradients.mean(dim=(2, 3), keepdim=True)
        cam = (alpha * activations).sum(dim=1)
        cam = F.relu(cam)

        heatmaps = []
        for t in range(seq_len):
            frame_cam = cam[t]
            frame_cam = frame_cam - frame_cam.min()
            max_val = frame_cam.max()
            if max_val > 1e-8:
                frame_cam = frame_cam / max_val
            heatmaps.append(frame_cam.detach().cpu().numpy())
        return np.stack(heatmaps), target_class, confidence


def overlay_heatmap_on_frame(frame_uint8, heatmap, alpha=0.45):
    H, W = frame_uint8.shape[:2]
    heatmap_resized = cv2.resize(heatmap, (W, H))
    heatmap_uint8 = (heatmap_resized * 255).astype(np.uint8)
    heatmap_color = cv2.applyColorMap(heatmap_uint8, cv2.COLORMAP_JET)
    heatmap_color = cv2.cvtColor(heatmap_color, cv2.COLOR_BGR2RGB)
    overlaid = (frame_uint8.astype(np.float32) * (1 - alpha) + heatmap_color.astype(np.float32) * alpha)
    return np.clip(overlaid, 0, 255).astype(np.uint8)


### 5.1 Generating Grad-CAM Visualizations

We apply Grad-CAM to the best-performing model (ResNet-18+LSTM), examining
two examples each of correctly and incorrectly classified sequences.

**Key finding (report Section 6.6):** across all examined sequences, the
model's spatial attention consistently localizes to background and
shoulder/clothing regions adjacent to the subject's face, rather than to
the eyes, mouth, or facial expression — direct visual evidence that the
model has learned to rely on subject/session-identifying visual cues
rather than genuine facial behavior.

In [ ]:
gradcam_model = build_cnn_lstm(backbone="resnet18", num_classes=2)
checkpoint = torch.load(f"{CHECKPOINT_DIR}/resnet18_lstm_best.pth", map_location=DEVICE)
gradcam_model.load_state_dict(checkpoint["model_state_dict"])
gradcam_model = gradcam_model.to(DEVICE)
gradcam_model.eval()

gradcam = SequenceGradCAM(gradcam_model, target_layer_name="cnn_backbone")

class_names = ["disengaged", "engaged"]
samples_shown = {0: 0, 1: 0}
max_per_class = 2

fig, axes = plt.subplots(max_per_class * 2, 8, figsize=(20, 10))
row_idx = 0

for idx in range(len(test_dataset)):
    seq_tensor, label = test_dataset[idx]
    if samples_shown[label] >= max_per_class:
        continue

    seq_batch = seq_tensor.unsqueeze(0).to(DEVICE)
    heatmaps, pred_class, confidence = gradcam.generate(seq_batch)

    clip_id, seq_idx, _ = test_dataset._index[idx]
    with h5py.File(test_dataset.h5_path, 'r') as hf:
        raw_sequence = hf['sequences'][clip_id][seq_idx]

    for t in range(8):
        overlay = overlay_heatmap_on_frame(raw_sequence[t], heatmaps[t])
        axes[row_idx, t].imshow(overlay)
        axes[row_idx, t].axis('off')
        if t == 0:
            correct = "✓" if pred_class == label else "✗"
            axes[row_idx, t].set_title(
                f"True: {class_names[label]}\nPred: {class_names[pred_class]} ({confidence:.2f}) {correct}",
                fontsize=9, loc='left')

    samples_shown[label] += 1
    row_idx += 1
    if all(v >= max_per_class for v in samples_shown.values()):
        break

plt.suptitle("Grad-CAM: Per-Frame Spatial Attention for ResNet-18+LSTM Predictions", fontsize=14)
plt.tight_layout()
plt.savefig(f"{PROJECT_BASE}/gradcam_visualization.png", dpi=150, bbox_inches='tight')
plt.show()

gradcam.remove_hooks()
print(f"\nSaved to {PROJECT_BASE}/gradcam_visualization.png")


## 6. Follow-Up Experiment: Full-Dataset Scaling

To directly test the hypothesis that the small number of training subjects
(69) was the primary bottleneck, we repeat all model trainings using
DAiSEE's full official training set (5,358 clips) rather than the 940-clip
stratified subsample. This requires re-extracting sequences for the full
manifest (reusing the same extraction logic from Part 1, Section 2.4).

**Critical bug encountered and fixed:** when reloading the manifest CSVs in
a fresh session, `clip_id` was silently inferred by pandas as `int64`
rather than `string`, because every clip ID happens to be purely numeric.
This caused HDF5 dataset-naming to fail downstream. We force the dtype
explicitly on load.

In [ ]:
# Reload manifests with explicit string dtype (fixes the int64 inference bug)
train_manifest_full = pd.read_csv(f"{PROJECT_BASE}/daisee_train_manifest.csv",
                                    dtype={'clip_id': str, 'participant_id': str})
val_manifest_full = pd.read_csv(f"{PROJECT_BASE}/daisee_val_manifest.csv",
                                  dtype={'clip_id': str, 'participant_id': str})
test_manifest_full = pd.read_csv(f"{PROJECT_BASE}/daisee_test_manifest.csv",
                                   dtype={'clip_id': str, 'participant_id': str})

print(f"Full manifests loaded: train={len(train_manifest_full)}, "
      f"val={len(val_manifest_full)}, test={len(test_manifest_full)}")
print("clip_id dtype:", train_manifest_full['clip_id'].dtype)


In [ ]:
# Extract sequences for the FULL dataset (reuses extract_sequences /
# build_sequence_hdf5 / write_metadata_table from Part 1)
for name, manifest_df in [("train", train_manifest_full), ("val", val_manifest_full), ("test", test_manifest_full)]:
    output_path = f"/content/daisee_{name}_full_sequences.h5"
    print(f"\nExtracting FULL sequences for {name} ({len(manifest_df)} clips), stride=16")
    report = build_sequence_hdf5(manifest_df, output_path, sequence_length=8, stride=16, progress_every=200)
    write_metadata_table(manifest_df, output_path)
    size_mb = os.path.getsize(output_path) / (1024**2)
    print(f"{name} report: {report}")
    print(f"HDF5 file size: {size_mb:.1f} MB")


In [ ]:
# Persist to Drive
for name in ["train", "val", "test"]:
    src = f"/content/daisee_{name}_full_sequences.h5"
    dst = f"{PROJECT_BASE}/daisee_{name}_full_sequences.h5"
    shutil.copy(src, dst)
    print(f"Saved {name}: {os.path.getsize(dst)/(1024**2):.1f} MB")

train_dataset_full = DAiSEESequenceDataset(f"{PROJECT_BASE}/daisee_train_full_sequences.h5", transform=build_frame_transform(train=True))
val_dataset_full = DAiSEESequenceDataset(f"{PROJECT_BASE}/daisee_val_full_sequences.h5", transform=build_frame_transform(train=False))
test_dataset_full = DAiSEESequenceDataset(f"{PROJECT_BASE}/daisee_test_full_sequences.h5", transform=build_frame_transform(train=False))

print(f"\nTrain: {len(train_dataset_full)} sequences, distribution: {train_dataset_full.get_label_distribution()}")
print(f"Val:   {len(val_dataset_full)} sequences, distribution: {val_dataset_full.get_label_distribution()}")
print(f"Test:  {len(test_dataset_full)} sequences, distribution: {test_dataset_full.get_label_distribution()}")


### 6.1 Training All Three Models on the Full Dataset

The full dataset's natural class imbalance (95.4% engaged, a 19.3:1 ratio)
is far more severe than the stratified subset's deliberately constructed
3:1 ratio. We use a 5x weight cap here (see Section 7 below for what
happens when this cap is removed).

In [ ]:
model_resnet_lstm_full = build_cnn_lstm(backbone="resnet18", num_classes=2)
ckpt_path_full, val_f1_full = train_cnn_lstm_capped(
    model_resnet_lstm_full, "resnet18_lstm_FULL",
    train_dataset_full, val_dataset_full,
    num_epochs=15, batch_size=32, patience=4
)


In [ ]:
def train_frame_classifier_capped(model, model_name, train_dataset_seq, val_dataset_seq,
                                     num_epochs=10, lr=1e-4, batch_size=32,
                                     max_weight_multiplier=5.0, patience=3):
    """Frame-level training using the capped-weighting scheme, operating on
    the FIRST frame of each sequence in a sequence-format HDF5 dataset."""
    model = model.to(DEVICE)

    class FrameWrapper(Dataset):
        def __init__(self, seq_dataset):
            self.h5_path = seq_dataset.h5_path
            self._index = seq_dataset._index
            self.transform = seq_dataset.transform

        def __len__(self):
            return len(self._index)

        def __getitem__(self, idx):
            clip_id, seq_idx, label = self._index[idx]
            with h5py.File(self.h5_path, 'r') as hf:
                frame = hf['sequences'][clip_id][seq_idx][0]
            frame_tensor = T.ToTensor()(frame)
            frame_tensor = T.Normalize(IMAGENET_MEAN, IMAGENET_STD)(frame_tensor)
            return frame_tensor, label

    frame_train = FrameWrapper(train_dataset_seq)
    frame_val = FrameWrapper(val_dataset_seq)
    train_loader = DataLoader(frame_train, batch_size=batch_size, shuffle=True, num_workers=2)
    val_loader = DataLoader(frame_val, batch_size=batch_size, shuffle=False, num_workers=2)

    dist = train_dataset_seq.get_label_distribution()
    weights = compute_capped_class_weights(dist, max_weight_multiplier).to(DEVICE)
    print(f"Class distribution: {dist}")
    print(f"Capped class weights: {weights.cpu().numpy()}")

    criterion = nn.CrossEntropyLoss(weight=weights)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=1e-4)

    best_val_f1 = 0.0
    epochs_no_improve = 0

    for epoch in range(1, num_epochs + 1):
        t0 = time.time()
        model.train()
        train_loss = 0.0
        for images, labels in train_loader:
            images, labels = images.to(DEVICE), labels.to(DEVICE)
            optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            train_loss += loss.item() * images.size(0)
        train_loss /= len(train_loader.dataset)

        model.eval()
        val_loss = 0.0
        all_preds, all_labels = [], []
        with torch.no_grad():
            for images, labels in val_loader:
                images, labels = images.to(DEVICE), labels.to(DEVICE)
                outputs = model(images)
                loss = criterion(outputs, labels)
                val_loss += loss.item() * images.size(0)
                preds = outputs.argmax(dim=1)
                all_preds.extend(preds.cpu().numpy())
                all_labels.extend(labels.cpu().numpy())
        val_loss /= len(val_loader.dataset)
        val_f1 = f1_score(all_labels, all_preds, average="macro")

        elapsed = time.time() - t0
        print(f"[{model_name}] Epoch {epoch:02d}/{num_epochs} | "
              f"train_loss={train_loss:.4f} val_loss={val_loss:.4f} val_macro_f1={val_f1:.4f} ({elapsed:.1f}s)")

        if val_f1 > best_val_f1:
            best_val_f1 = val_f1
            epochs_no_improve = 0
            torch.save(model.state_dict(), f"{CHECKPOINT_DIR}/{model_name}_best.pth")
            print(f"  -> New best (val_macro_f1={val_f1:.4f})")
        else:
            epochs_no_improve += 1
            if epochs_no_improve >= patience:
                print(f"  -> Early stopping at epoch {epoch}")
                break

    print(f"\n[{model_name}] Best val macro-F1: {best_val_f1:.4f}")
    return best_val_f1


frame_model_full = build_frame_classifier()
best_f1_frame_full = train_frame_classifier_capped(
    frame_model_full, "mobilenetv3_framelevel_FULL",
    train_dataset_full, val_dataset_full,
    num_epochs=10, batch_size=64
)


In [ ]:
model_mobilenet_lstm_full = build_cnn_lstm(backbone="mobilenetv3", num_classes=2)
ckpt_path_mob_full, val_f1_mob_full = train_cnn_lstm_capped(
    model_mobilenet_lstm_full, "mobilenetv3_lstm_FULL",
    train_dataset_full, val_dataset_full,
    num_epochs=15, batch_size=32, patience=4
)


### 6.2 Test Set Evaluation — Full Dataset Models

In [ ]:
def evaluate_on_test_full(model, checkpoint_path, test_dataset, model_name,
                            batch_size=32, is_frame_level=False):
    checkpoint = torch.load(checkpoint_path, map_location=DEVICE)
    if isinstance(checkpoint, dict) and "model_state_dict" in checkpoint:
        model.load_state_dict(checkpoint["model_state_dict"])
    else:
        model.load_state_dict(checkpoint)
    model = model.to(DEVICE)
    model.eval()

    if is_frame_level:
        class FrameWrapper(Dataset):
            def __init__(self, seq_dataset):
                self.h5_path = seq_dataset.h5_path
                self._index = seq_dataset._index
            def __len__(self):
                return len(self._index)
            def __getitem__(self, idx):
                clip_id, seq_idx, label = self._index[idx]
                with h5py.File(self.h5_path, 'r') as hf:
                    frame = hf['sequences'][clip_id][seq_idx][0]
                frame_tensor = T.ToTensor()(frame)
                frame_tensor = T.Normalize(IMAGENET_MEAN, IMAGENET_STD)(frame_tensor)
                return frame_tensor, label
        eval_dataset = FrameWrapper(test_dataset)
    else:
        eval_dataset = test_dataset

    test_loader = DataLoader(eval_dataset, batch_size=batch_size, shuffle=False, num_workers=2)

    all_preds, all_labels = [], []
    with torch.no_grad():
        for inputs, labels in test_loader:
            inputs = inputs.to(DEVICE)
            outputs = model(inputs)
            preds = outputs.argmax(dim=1)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.numpy())

    acc = (np.array(all_preds) == np.array(all_labels)).mean()
    macro_f1 = f1_score(all_labels, all_preds, average="macro")

    print(f"\n{'='*60}\n{model_name} — TEST SET RESULTS\n{'='*60}")
    print(f"Accuracy: {acc:.4f}\nMacro-F1: {macro_f1:.4f}")
    print(classification_report(all_labels, all_preds, target_names=["disengaged", "engaged"], digits=3))

    return {"model_name": model_name, "test_accuracy": acc, "test_macro_f1": macro_f1}


results_full = []
frame_model_full_eval = build_frame_classifier()
results_full.append(evaluate_on_test_full(frame_model_full_eval, f"{CHECKPOINT_DIR}/mobilenetv3_framelevel_FULL_best.pth",
                                            test_dataset_full, "MobileNetV3 (frame-level) - FULL", is_frame_level=True))

model_mobilenet_lstm_full_eval = build_cnn_lstm(backbone="mobilenetv3", num_classes=2)
results_full.append(evaluate_on_test_full(model_mobilenet_lstm_full_eval, f"{CHECKPOINT_DIR}/mobilenetv3_lstm_FULL_best.pth",
                                            test_dataset_full, "MobileNetV3 + LSTM - FULL"))

model_resnet_lstm_full_eval = build_cnn_lstm(backbone="resnet18", num_classes=2)
results_full.append(evaluate_on_test_full(model_resnet_lstm_full_eval, f"{CHECKPOINT_DIR}/resnet18_lstm_FULL_best.pth",
                                            test_dataset_full, "ResNet-18 + LSTM - FULL"))

print("\n\n" + "="*60 + "\nFULL-DATASET TEST SET COMPARISON\n" + "="*60)
summary_df_full = pd.DataFrame(results_full)[["model_name", "test_accuracy", "test_macro_f1"]]
print(summary_df_full.to_string(index=False))
summary_df_full.to_csv(f"{PROJECT_BASE}/final_test_results_FULL.csv", index=False)


## 7. Follow-Up: Uncapped Class Weighting

Since the full-dataset models showed near-total failure to recognize the
minority disengaged class (recall as low as 4.6%), we test whether this
was caused by the 5x weight cap being too weak against the true 19.3:1
imbalance. We retrain ResNet-18+LSTM with weights matching the true ratio
(in practice ~20.7x, since weights are normalized as inverse frequency).

**Result (report Section 6.7.1):** disengaged recall more than doubled
(6.0% → 13.5%), but macro-F1 barely changed, because precision fell
correspondingly. This demonstrates that class-weighting is necessary but
not sufficient — the deeper bottleneck is representational, not just a
loss-function calibration issue (consistent with the Grad-CAM evidence in
Section 5 above).

In [ ]:
model_resnet_lstm_uncapped = build_cnn_lstm(backbone="resnet18", num_classes=2)
ckpt_path_uncapped, val_f1_uncapped = train_cnn_lstm_capped(
    model_resnet_lstm_uncapped, "resnet18_lstm_FULL_uncapped",
    train_dataset_full, val_dataset_full,
    num_epochs=15, batch_size=32, patience=4,
    max_weight_multiplier=25.0  # set above the true ~20.7x ratio so the cap never triggers
)


In [ ]:
model_uncapped_eval = build_cnn_lstm(backbone="resnet18", num_classes=2)
result_uncapped = evaluate_on_test(
    model_uncapped_eval,
    f"{CHECKPOINT_DIR}/resnet18_lstm_FULL_uncapped_best.pth",
    test_dataset_full,
    "ResNet-18 + LSTM (full dataset, uncapped weighting)",
    class_names=("disengaged", "engaged")
)


## 8. Follow-Up: 3-Class Extension (Engaged / Bored / Other)

To rule out the binary framing itself as the source of the limited
performance, we extend to a 3-class formulation using DAiSEE's four raw
ordinal annotation dimensions, taking the argmax across
{boredom, engagement, confusion, frustration} per clip and collapsing
Confusion and Frustration into a single "Other" category due to their
extreme scarcity (combined ~1.6% of clips).

**Tie-resolution methodology (a real data-quality finding):** 20–30% of
clips have a tie at the maximum score across two or more dimensions.
3-way and 4-way ties (clips with no genuine differentiated signal, most
commonly all four dimensions rated equally) are dropped entirely. The
common 2-way Boredom/Engagement tie is resolved using each clip's
already-validated `engagement_binary` label — NOT an arbitrary
column-ordering default, which would otherwise systematically
under-represent Boredom in roughly half of all tied clips.

In [ ]:
LABEL_COLUMNS = ['boredom', 'engagement', 'confusion', 'frustration']

def build_three_class_labels(df):
    """
    Assigns a 3-class label (engaged/bored/other) via argmax across the
    four raw DAiSEE dimensions. Drops clips with 3-way/4-way ties (no
    differentiated signal); resolves 2-way Boredom/Engagement ties using
    the validated engagement_binary label rather than column order.
    """
    df = df.copy()

    def get_tied_dims(row):
        scores = row[LABEL_COLUMNS]
        max_val = scores.max()
        return tuple(sorted(scores[scores == max_val].index.tolist()))

    df['_tied_dims'] = df.apply(get_tied_dims, axis=1)
    df['_tie_size'] = df['_tied_dims'].apply(len)

    original_count = len(df)
    df_clean = df[df['_tie_size'] <= 2].copy()
    dropped_count = original_count - len(df_clean)

    def assign_label(row):
        tied = row['_tied_dims']
        if len(tied) == 1:
            dim = tied[0]
        else:
            if set(tied) == {'boredom', 'engagement'}:
                dim = 'engagement' if row['engagement_binary'] == 1 else 'boredom'
            elif 'confusion' in tied or 'frustration' in tied:
                return 'other'
            else:
                dim = tied[0]
        if dim == 'engagement':
            return 'engaged'
        elif dim == 'boredom':
            return 'bored'
        else:
            return 'other'

    df_clean['three_class_label'] = df_clean.apply(assign_label, axis=1)
    df_clean = df_clean.drop(columns=['_tied_dims', '_tie_size'])

    report = {
        "original_count": original_count,
        "dropped_3way_4way_ties": dropped_count,
        "final_count": len(df_clean),
        "label_distribution": df_clean['three_class_label'].value_counts().to_dict()
    }
    return df_clean, report


train_3class, train_report = build_three_class_labels(train_subset)
val_3class, val_report = build_three_class_labels(val_subset)
test_3class, test_report = build_three_class_labels(test_subset)

for name, report in [("Train", train_report), ("Val", val_report), ("Test", test_report)]:
    print(f"{name}: {report}")

train_3class.to_csv(f"{PROJECT_BASE}/daisee_train_3class.csv", index=False)
val_3class.to_csv(f"{PROJECT_BASE}/daisee_val_3class.csv", index=False)
test_3class.to_csv(f"{PROJECT_BASE}/daisee_test_3class.csv", index=False)


In [ ]:
# Relabel the EXISTING binary HDF5 sequences with 3-class labels
# (avoids re-extracting video frames -- just copies sequence arrays + new metadata)
def relabel_hdf5_to_3class(source_h5_path, target_h5_path, three_class_manifest):
    label_map = {"engaged": 0, "bored": 1, "other": 2}
    clip_to_label = dict(zip(three_class_manifest['clip_id'],
                               three_class_manifest['three_class_label'].map(label_map)))

    with h5py.File(source_h5_path, 'r') as src, h5py.File(target_h5_path, 'w') as dst:
        src_seq_group = src['sequences']
        dst_seq_group = dst.create_group('sequences')
        copied = 0
        for clip_id in src_seq_group.keys():
            if clip_id in clip_to_label:
                dst_seq_group.create_dataset(clip_id, data=src_seq_group[clip_id][:],
                                               compression='gzip', compression_opts=4)
                copied += 1

        meta_group = dst.create_group('metadata')
        string_dtype = h5py.special_dtype(vlen=str)
        clip_ids = list(clip_to_label.keys())
        labels_3class = [clip_to_label[c] for c in clip_ids]
        meta_group.create_dataset('clip_id', data=clip_ids, dtype=string_dtype)
        meta_group.create_dataset('three_class_label', data=labels_3class)

    print(f"Copied {copied} clips from {source_h5_path} to {target_h5_path}")
    return copied


for name, manifest_3class in [("train", train_3class), ("val", val_3class), ("test", test_3class)]:
    source = f"{PROJECT_BASE}/daisee_{name}_sequences.h5"
    target = f"/content/daisee_{name}_3class_sequences.h5"
    relabel_hdf5_to_3class(source, target, manifest_3class)
    shutil.copy(target, f"{PROJECT_BASE}/daisee_{name}_3class_sequences.h5")


In [ ]:
class DAiSEE3ClassDataset(Dataset):
    """3-class dataset (three_class_label: 0=engaged, 1=bored, 2=other)."""
    def __init__(self, h5_path, transform=None):
        self.h5_path = h5_path
        self.transform = transform
        self._index = []
        with h5py.File(h5_path, 'r') as hf:
            seq_group = hf['sequences']
            meta = hf['metadata']
            clip_ids = [c if isinstance(c, str) else c.decode() for c in meta['clip_id'][:]]
            labels = meta['three_class_label'][:]
            label_lookup = dict(zip(clip_ids, labels))
            for clip_id in seq_group.keys():
                num_sequences = seq_group[clip_id].shape[0]
                label = int(label_lookup[clip_id])
                for seq_idx in range(num_sequences):
                    self._index.append((clip_id, seq_idx, label))

    def __len__(self):
        return len(self._index)

    def __getitem__(self, idx):
        clip_id, seq_idx, label = self._index[idx]
        with h5py.File(self.h5_path, 'r') as hf:
            sequence = hf['sequences'][clip_id][seq_idx]
        if self.transform:
            sequence_tensor = self.transform(sequence)
        else:
            sequence_tensor = torch.from_numpy(sequence).float() / 255.0
            sequence_tensor = sequence_tensor.permute(0, 3, 1, 2)
        return sequence_tensor, label

    def get_label_distribution(self):
        labels = [item[2] for item in self._index]
        unique, counts = np.unique(labels, return_counts=True)
        return dict(zip(unique.tolist(), counts.tolist()))


train_dataset_3class = DAiSEE3ClassDataset(f"{PROJECT_BASE}/daisee_train_3class_sequences.h5", transform=build_frame_transform(train=True))
val_dataset_3class = DAiSEE3ClassDataset(f"{PROJECT_BASE}/daisee_val_3class_sequences.h5", transform=build_frame_transform(train=False))
test_dataset_3class = DAiSEE3ClassDataset(f"{PROJECT_BASE}/daisee_test_3class_sequences.h5", transform=build_frame_transform(train=False))

print(f"Train: {len(train_dataset_3class)} sequences, distribution: {train_dataset_3class.get_label_distribution()}")
print(f"Val:   {len(val_dataset_3class)} sequences, distribution: {val_dataset_3class.get_label_distribution()}")
print(f"Test:  {len(test_dataset_3class)} sequences, distribution: {test_dataset_3class.get_label_distribution()}")


In [ ]:
def train_cnn_lstm_3class(model, model_name, train_dataset, val_dataset,
                            num_epochs=15, lr=1e-4, weight_decay=1e-4,
                            batch_size=16, patience=4):
    """Training loop for the 3-class model, using standard (uncapped)
    inverse-frequency weighting -- the natural imbalance here (max ratio
    ~7x) is mild enough that no capping is necessary."""
    model = model.to(DEVICE)
    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=2)
    val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False, num_workers=2)

    dist = train_dataset.get_label_distribution()
    total = sum(dist.values())
    num_classes = len(dist)
    weights = torch.tensor([total / dist[i] for i in range(num_classes)], dtype=torch.float32)
    weights = weights / weights.sum() * num_classes
    weights = weights.to(DEVICE)
    print(f"Class distribution: {dist}")
    print(f"Class weights: {weights.cpu().numpy()}")

    criterion = nn.CrossEntropyLoss(weight=weights)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)

    best_val_f1 = 0.0
    epochs_no_improve = 0

    for epoch in range(1, num_epochs + 1):
        t0 = time.time()
        model.train()
        train_loss = 0.0
        for sequences, labels in train_loader:
            sequences, labels = sequences.to(DEVICE), labels.to(DEVICE)
            optimizer.zero_grad()
            outputs = model(sequences)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            train_loss += loss.item() * sequences.size(0)
        train_loss /= len(train_loader.dataset)

        model.eval()
        val_loss = 0.0
        all_preds, all_labels = [], []
        with torch.no_grad():
            for sequences, labels in val_loader:
                sequences, labels = sequences.to(DEVICE), labels.to(DEVICE)
                outputs = model(sequences)
                loss = criterion(outputs, labels)
                val_loss += loss.item() * sequences.size(0)
                preds = outputs.argmax(dim=1)
                all_preds.extend(preds.cpu().numpy())
                all_labels.extend(labels.cpu().numpy())
        val_loss /= len(val_loader.dataset)
        val_f1 = f1_score(all_labels, all_preds, average="macro")

        elapsed = time.time() - t0
        print(f"[{model_name}] Epoch {epoch:02d}/{num_epochs} | "
              f"train_loss={train_loss:.4f} val_loss={val_loss:.4f} val_macro_f1={val_f1:.4f} ({elapsed:.1f}s)")

        if val_f1 > best_val_f1:
            best_val_f1 = val_f1
            epochs_no_improve = 0
            torch.save({"model_state_dict": model.state_dict(),
                        "epoch": epoch, "val_f1": val_f1}, f"{CHECKPOINT_DIR}/{model_name}_best.pth")
            print(f"  -> New best model saved (val_macro_f1={val_f1:.4f})")
        else:
            epochs_no_improve += 1
            if epochs_no_improve >= patience:
                print(f"  -> Early stopping at epoch {epoch}")
                break

    print(f"\n[{model_name}] Best validation macro-F1: {best_val_f1:.4f}")
    return f"{CHECKPOINT_DIR}/{model_name}_best.pth", best_val_f1


model_resnet_lstm_3class = build_cnn_lstm(backbone="resnet18", num_classes=3)
ckpt_path_3class, val_f1_3class = train_cnn_lstm_3class(
    model_resnet_lstm_3class, "resnet18_lstm_3class",
    train_dataset_3class, val_dataset_3class,
    num_epochs=15, batch_size=32, patience=4
)


In [ ]:
model_3class_eval = build_cnn_lstm(backbone="resnet18", num_classes=3)
result_3class = evaluate_on_test(
    model_3class_eval, f"{CHECKPOINT_DIR}/resnet18_lstm_3class_best.pth",
    test_dataset_3class, "ResNet-18 + LSTM (3-class)",
    class_names=("engaged", "bored", "other")
)

pd.DataFrame([result_3class])[["model_name", "test_accuracy", "test_macro_f1"]].to_csv(
    f"{PROJECT_BASE}/final_test_results_3class.csv", index=False)


## Summary

This notebook implements the complete experimental pipeline described in
the project report:

| Experiment | Report Section | Key Result |
|---|---|---|
| Frame-level vs. CNN+LSTM (subset) | 6.1–6.5 | ResNet-18+LSTM best, macro-F1 0.595 |
| Grad-CAM explainability | 6.6 | Model attends to background, not faces |
| Full-dataset scaling | 6.7 | Scaling alone did not help; minority recall collapsed |
| Uncapped class weighting | 6.7.1 | Recall improved 6.0%→13.5%, macro-F1 unchanged |
| 3-class extension | 6.8 | Same pattern reproduced (macro-F1 0.324, near chance) |

See the accompanying report for full discussion and conclusions.

## 9. Fetch Saved Results and Regenerate Report Figures

**Use this cell if you have already run the notebook once and just want to
reload your results** (training logs, test-set metrics) and regenerate the
report figures, without re-running any training.

This cell only READS from Drive — it does not retrain or re-extract
anything. It expects the following to already exist under
`{PROJECT_BASE}/logs/` and `{PROJECT_BASE}/*.csv` from a previous full run
of this notebook:

- Training logs: `mobilenetv3_lstm`, `resnet18_lstm`, `resnet18_lstm_frozen`,
  `mobilenetv3_lstm_FULL`, `resnet18_lstm_FULL`,
  `resnet18_lstm_FULL_uncapped`, `resnet18_lstm_3class`
  (each as `{model_name}_training_log.csv`)
- Test-set result CSVs: `final_test_results_all4.csv`,
  `final_test_results_FULL.csv`, `final_test_results_3class.csv`

If any of these are missing, this cell will tell you exactly which one and
you'll need to re-run the corresponding training cell earlier in this
notebook (no need to re-run everything).

In [ ]:
# ============================================================
# FETCH SAVED RESULTS (no training/extraction required)
# ============================================================
from google.colab import drive
drive.mount('/content/drive')

import os
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')  # safe default; Colab will still display inline below
import matplotlib.pyplot as plt

PROJECT_BASE = '/content/drive/MyDrive/DLCV_Project'
LOG_DIR = f"{PROJECT_BASE}/logs"
FIGURES_DIR = f"{PROJECT_BASE}/figures"
os.makedirs(FIGURES_DIR, exist_ok=True)

plt.rcParams.update({
    'font.size': 11, 'font.family': 'sans-serif',
    'axes.spines.top': False, 'axes.spines.right': False,
    'axes.grid': True, 'grid.alpha': 0.25,
    'figure.facecolor': 'white', 'axes.facecolor': 'white',
})
BLUE, RED, GREEN, ORANGE, PURPLE, GREY = (
    "#1F4E79", "#C0392B", "#27AE60", "#E67E22", "#8E44AD", "#7F8C8D")


def load_log(model_name):
    """Loads one training log CSV, or returns None with a clear warning if missing."""
    path = f"{LOG_DIR}/{model_name}_training_log.csv"
    if not os.path.exists(path):
        print(f"  MISSING: {path} -- re-run the training cell for '{model_name}' earlier in this notebook")
        return None
    df = pd.read_csv(path)
    print(f"  Loaded {model_name}: {len(df)} epochs")
    return df


print("=" * 60)
print("Loading training logs...")
print("=" * 60)
log_names = ["mobilenetv3_lstm", "resnet18_lstm", "resnet18_lstm_frozen",
             "mobilenetv3_lstm_FULL", "resnet18_lstm_FULL",
             "resnet18_lstm_FULL_uncapped", "resnet18_lstm_3class"]
logs = {name: load_log(name) for name in log_names}

print("\n" + "=" * 60)
print("Loading test-set result CSVs...")
print("=" * 60)
result_files = {
    "subset_4models": "final_test_results_all4.csv",
    "full_dataset": "final_test_results_FULL.csv",
    "three_class": "final_test_results_3class.csv",
    "uncapped": "final_test_results.csv",  # may not exist under this exact name -- checked below
}
results = {}
for key, fname in result_files.items():
    path = f"{PROJECT_BASE}/{fname}"
    if os.path.exists(path):
        results[key] = pd.read_csv(path)
        print(f"  Loaded {fname}:")
        print(results[key].to_string(index=False))
        print()
    else:
        print(f"  Not found (may not have been saved under this name): {fname}")

print("\nAll available results loaded. Proceeding to regenerate figures from logs...")


In [ ]:
# ============================================================
# REGENERATE FIGURES from the loaded logs (matches report figures 1, 3, 4, 7)
# ============================================================

def plot_training_curves(ax, ax2, df, title):
    """Plots train/val loss + val macro-F1 on a given axis pair."""
    l1, = ax.plot(df["epoch"], df["train_loss"], color=BLUE, marker='o', markersize=4, label="Train loss")
    l2, = ax.plot(df["epoch"], df["val_loss"], color=RED, marker='s', markersize=4, label="Val loss")
    l3, = ax2.plot(df["epoch"], df["val_macro_f1"], color=GREEN, marker='^', markersize=4,
                    linestyle='--', label="Val macro-F1")
    best_idx = df["val_macro_f1"].idxmax()
    ax2.scatter([df["epoch"][best_idx]], [df["val_macro_f1"][best_idx]], s=120,
                facecolors='none', edgecolors=GREEN, linewidths=2, zorder=5)
    ax.set_xlabel("Epoch")
    ax.set_ylabel("Loss", color=BLUE)
    ax2.set_ylabel("Val Macro-F1", color=GREEN)
    ax.set_title(title, fontsize=11.5, fontweight='bold')
    ax.tick_params(axis='y', labelcolor=BLUE)
    ax2.tick_params(axis='y', labelcolor=GREEN)
    ax.set_xticks(df["epoch"])
    return l1, l2, l3


# --- Figure: subset models (3-panel) ---
panel_configs = [("mobilenetv3_lstm", "MobileNetV3 + LSTM"),
                  ("resnet18_lstm", "ResNet-18 + LSTM"),
                  ("resnet18_lstm_frozen", "ResNet-18 + LSTM (frozen)")]
available_panels = [(k, t) for k, t in panel_configs if logs.get(k) is not None]

if available_panels:
    fig, axes = plt.subplots(1, len(available_panels), figsize=(5 * len(available_panels), 4.2))
    if len(available_panels) == 1:
        axes = [axes]
    for ax, (key, title) in zip(axes, available_panels):
        ax2 = ax.twinx()
        l1, l2, l3 = plot_training_curves(ax, ax2, logs[key], title)
    fig.legend([l1, l2, l3], ["Train loss", "Val loss", "Val macro-F1 (circled = best epoch)"],
               loc='lower center', ncol=3, bbox_to_anchor=(0.5, -0.08), frameon=False)
    fig.suptitle("Subset Models: Training Dynamics (regenerated)", fontsize=13, y=1.04)
    plt.tight_layout()
    plt.savefig(f"{FIGURES_DIR}/regenerated_subset_training_curves.png", dpi=150, bbox_inches='tight')
    plt.show()
    print(f"Saved: {FIGURES_DIR}/regenerated_subset_training_curves.png")
else:
    print("Skipping subset training-curve figure -- no relevant logs found.")


# --- Figure: capped vs uncapped full-dataset ---
full_configs = [("resnet18_lstm_FULL", "Full Dataset (5x cap)"),
                 ("resnet18_lstm_FULL_uncapped", "Full Dataset (uncapped)")]
available_full = [(k, t) for k, t in full_configs if logs.get(k) is not None]

if available_full:
    fig, axes = plt.subplots(1, len(available_full), figsize=(5.5 * len(available_full), 4.2))
    if len(available_full) == 1:
        axes = [axes]
    for ax, (key, title) in zip(axes, available_full):
        ax2 = ax.twinx()
        l1, l2, l3 = plot_training_curves(ax, ax2, logs[key], title)
    fig.legend([l1, l2, l3], ["Train loss", "Val loss", "Val macro-F1"],
               loc='lower center', ncol=3, bbox_to_anchor=(0.5, -0.1), frameon=False)
    fig.suptitle("Full-Dataset: Capped vs. Uncapped Weighting (regenerated)", fontsize=13, y=1.03)
    plt.tight_layout()
    plt.savefig(f"{FIGURES_DIR}/regenerated_capped_vs_uncapped.png", dpi=150, bbox_inches='tight')
    plt.show()
    print(f"Saved: {FIGURES_DIR}/regenerated_capped_vs_uncapped.png")
else:
    print("Skipping capped-vs-uncapped figure -- no relevant logs found.")


# --- Figure: 3-class training curve ---
if logs.get("resnet18_lstm_3class") is not None:
    df = logs["resnet18_lstm_3class"]
    fig, ax = plt.subplots(figsize=(6.5, 4.2))
    ax2 = ax.twinx()
    l1, l2, l3 = plot_training_curves(ax, ax2, df, "3-Class Model Training Dynamics")
    l4 = ax2.axhline(0.333, color=GREY, linestyle=':', linewidth=1.5)
    fig.legend([l1, l2, l3, l4], ["Train loss", "Val loss", "Val macro-F1", "Chance level (1/3)"],
               loc='lower center', ncol=4, bbox_to_anchor=(0.5, -0.18), frameon=False)
    plt.tight_layout()
    plt.savefig(f"{FIGURES_DIR}/regenerated_3class_training_curve.png", dpi=150, bbox_inches='tight')
    plt.show()
    print(f"Saved: {FIGURES_DIR}/regenerated_3class_training_curve.png")
else:
    print("Skipping 3-class training-curve figure -- log not found.")


# --- Figure: headline macro-F1 bar chart (from saved test-result CSVs) ---
all_results_rows = []
for key in ["subset_4models", "full_dataset", "three_class"]:
    if key in results:
        all_results_rows.append(results[key])

if all_results_rows:
    combined = pd.concat(all_results_rows, ignore_index=True)
    fig, ax = plt.subplots(figsize=(13, 5.5))
    ax.grid(axis='y', alpha=0.25)
    ax.set_axisbelow(True)
    bars = ax.bar(range(len(combined)), combined["test_macro_f1"], color=BLUE, edgecolor='white')
    for bar, val in zip(bars, combined["test_macro_f1"]):
        ax.text(bar.get_x() + bar.get_width()/2, val + 0.012, f"{val:.3f}",
                ha='center', va='bottom', fontsize=9, fontweight='bold')
    ax.set_xticks(range(len(combined)))
    ax.set_xticklabels(combined["model_name"], rotation=30, ha='right', fontsize=8.5)
    ax.set_ylabel("Test Macro-F1")
    ax.set_title("All Models: Test Macro-F1 (regenerated from saved results)", fontsize=13, fontweight='bold')
    plt.tight_layout()
    plt.savefig(f"{FIGURES_DIR}/regenerated_all_models_macro_f1.png", dpi=150, bbox_inches='tight')
    plt.show()
    print(f"Saved: {FIGURES_DIR}/regenerated_all_models_macro_f1.png")
else:
    print("Skipping headline bar chart -- no test-result CSVs found.")

print("\n" + "=" * 60)
print("DONE. Regenerated figures saved to:", FIGURES_DIR)
print("=" * 60)
